In [3]:
import os
import sys

# Get the absolute path to the project root (two levels up from this notebook)
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

# Add it to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

# Linear Regression

You might have thought to yourself : “What are these functions ? where do they come from ?”, and you would be right to skip them. But you are missing on a very good opportunity to better understand calculus and the world of derivation.

We have our linear function that we want to fit to the values (it is a line)

$$
\^{y} = wx+b\\
x : variable\\
w : weight\\
b : bias\ or\ intercept \\
\^{y} : result 
$$

To know how much off we are, we are going to use the Sum Squared Error

$$
J(w,b)=\displaystyle\sum_{i=1}^n(y-\^{y})^2 \\
$$

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# 1. Initialize data
df = pd.DataFrame({"km": [0.5, 2.0, 2.9], "price": [1.0, 1.90, 3.25]})

# 2. Pre-calculate the Cost Surface
w_range = np.linspace(-1, 3, 100)
b_range = np.linspace(-1, 3, 100)
W, B = np.meshgrid(w_range, b_range)
Z = np.zeros_like(W)

for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        y_preds = W[i, j] * df["km"] + B[i, j]
        Z[i, j] = np.sum((y_preds - df["price"]) ** 2)


# 3. Define the update function
def plot_regression(w, b):
    # Create 2x2 subplots
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))

    # --- TOP-LEFT: Regression Line ---
    ax1 = axs[0, 0]
    ax1.scatter(df["km"], df["price"], color="blue", s=100, label="Data")
    x_line = np.linspace(0, 3.5, 100)
    y_line = w * x_line + b
    ax1.plot(x_line, y_line, color="red", linewidth=2, label="Prediction")

    current_preds = w * df["km"] + b
    residuals = current_preds - df["price"]
    current_sse = np.sum(residuals**2)

    for i in range(len(df)):
        ax1.vlines(
            df["km"][i],
            df["price"][i],
            current_preds[i],
            colors="gray",
            linestyles="--",
            alpha=0.5,
        )
    ax1.set_title(f"Model Fit\nSum Squared Error: {current_sse:.4f}")
    ax1.set_xlabel("Km Driven")
    ax1.set_ylabel("Price")
    ax1.set_xlim(0, 3.5)
    ax1.set_ylim(0, 4)
    ax1.legend()
    ax1.grid(True, linestyle=":", alpha=0.6)

    # --- TOP-RIGHT: Cost Landscape (Contour) ---
    ax2 = axs[0, 1]
    contour = ax2.contourf(W, B, Z, levels=25, cmap="viridis")
    fig.colorbar(contour, ax=ax2, label="SSE Cost")
    ax2.scatter(
        w,
        b,
        c="red",
        s=100,
        edgecolors="white",
        linewidth=2,
        zorder=10,
        label="Current Param",
    )
    ax2.set_title("Cost Function Landscape (SSE)")
    ax2.set_xlabel("Weight (w)")
    ax2.set_ylabel("Bias (b)")
    ax2.legend()
    ax2.grid(True, linestyle=":", alpha=0.3)

    # --- Calculate Derivatives (Gradients) for Tangents ---
    # dJ/dw = sum(2 * (y_pred - y_real) * x)
    # dJ/db = sum(2 * (y_pred - y_real))
    grad_w = np.sum(2 * residuals * df["km"])
    grad_b = np.sum(2 * residuals)

    # --- BOTTOM-LEFT: Cost vs Weight (w) ---
    ax3 = axs[1, 0]
    sse_w = [np.sum(((w_val * df["km"] + b) - df["price"]) ** 2) for w_val in w_range]

    ax3.plot(w_range, sse_w, color="green", linewidth=2, label="Cost Curve")
    ax3.scatter(w, current_sse, c="red", s=100, zorder=10, label="Current w")

    # Plot Tangent Line for w
    # y = slope * (x - x0) + y0
    tangent_w = grad_w * (w_range - w) + current_sse
    ax3.plot(
        w_range,
        tangent_w,
        color="orange",
        linestyle="--",
        linewidth=1.5,
        label=f"Tangent (slope={grad_w:.2f})",
    )

    ax3.set_title(f"Cost vs Weight (w) | fixed b={b:.2f}")
    ax3.set_xlabel("Weight (w)")
    ax3.set_ylabel("SSE")
    ax3.set_ylim(0, max(sse_w) * 1.1)
    ax3.grid(True, linestyle=":", alpha=0.6)
    ax3.legend()

    # --- BOTTOM-RIGHT: Cost vs Bias (b) ---
    ax4 = axs[1, 1]
    sse_b = [np.sum(((w * df["km"] + b_val) - df["price"]) ** 2) for b_val in b_range]

    ax4.plot(b_range, sse_b, color="purple", linewidth=2, label="Cost Curve")
    ax4.scatter(b, current_sse, c="red", s=100, zorder=10, label="Current b")

    # Plot Tangent Line for b
    tangent_b = grad_b * (b_range - b) + current_sse
    ax4.plot(
        b_range,
        tangent_b,
        color="orange",
        linestyle="--",
        linewidth=1.5,
        label=f"Tangent (slope={grad_b:.2f})",
    )

    ax4.set_title(f"Cost vs Bias (b) | fixed w={w:.2f}")
    ax4.set_xlabel("Bias (b)")
    ax4.set_ylabel("SSE")
    ax4.set_ylim(0, max(sse_b) * 1.1)
    ax4.grid(True, linestyle=":", alpha=0.6)
    ax4.legend()

    plt.tight_layout()
    plt.show()


# 4. Create Interactive Sliders
w_slider = widgets.FloatSlider(
    value=0.5, min=-1.0, max=3.0, step=0.05, description="w (Weight):"
)
b_slider = widgets.FloatSlider(
    value=0.0, min=-1.0, max=3.0, step=0.05, description="b (Bias):"
)

ui = widgets.VBox([w_slider, b_slider])
out = widgets.interactive_output(plot_regression, {"w": w_slider, "b": b_slider})

display(ui, out)

Output()

When you move the Weight ($w$) slider, notice that the curve in the bottom-right plot (Cost vs Bias) actually shifts up and down or changes shape.

Because the "best" height ($b$) keeps shifting depending on the angle ($w$), we can't just find the perfect $b$ once and forget about it. 
<br>We have to constantly adjust both at the same time to walk down to the bottom of that bowl.

This process is exactly what Gradient Descent performs.
<br>It looks at the slope of the error landscape (the derivatives you calculated) to figure out which direction is "downhill" for both parameters simultaneously.

## Sum of Squared Error

$$
J(w,b)=\displaystyle\sum_{i=1}^n(y-\^{y})^2 \\
\^{y} : expected \ result

$$

### Derive in respect of b
Say : $J(w,b) = f$

We want the minimum error for $f$, we want $\delta f = 0$

We are first going to optimise for bias → $b$ 

$$
f(g(b))
$$

$$
f(g)=g^2\\
g(b)=y-\^{y} \equiv y-wx-b\\ \\
$$

First, let’s differentiate each function



for $f$, we know that :

$$
\delta f = \delta f(g)\delta g\\
\equiv \dfrac{\delta f}{\delta g} = \delta f(g)
$$

So in our case :

$$
\delta f = \delta g^2\delta g\\
\equiv \delta f = 2g\delta g\\
\equiv \dfrac{\delta f}{\delta g} = 2g
$$

Note : $\delta g$ is negligible (3B1B visual explanation [https://www.youtube.com/watch?v=S0_qX4VJhMQ](https://www.youtube.com/watch?v=S0_qX4VJhMQ))

And for g :

$$
\delta g = \delta g(b)\delta b\\
\equiv \dfrac{\delta g}{\delta b} = \delta g(b)
$$

$$
\delta g = \delta (y - (wx+b))\delta b \\
\equiv \delta g = \delta (y -wx -b) \delta b \\
\equiv \delta g = -1\delta b \\
\equiv \dfrac{\delta g}{\delta b} = -1
$$

$$
\frac{\delta}{\delta b}(y) = 0 \\
\frac{\delta}{\delta b}(-wx) = 0 \\
\frac{\delta}{\delta b}(-b) = -1 \\

$$

We have a composite function, we have to apply the chain rule (to differentiate the outer and inner functions) to find the derivative of $f$ with respect of $b$

[Chain Rule](https://www.youtube.com/watch?v=wl1myxrtQHQ) and [vizualization](https://www.youtube.com/watch?v=YG15m2VwSjA)

$$
\delta f(g(b))= \delta f(g) \delta g(b) \delta b \\
\equiv \dfrac {\delta f(g(b))}{\delta b} = \delta f(g) \delta g(b) \\
\equiv \dfrac {\delta f(g(b))}{\delta b} = -2(y - wx+b) \\
$$

### Derive in respect of w


Then we are going to optimise for weight → $w$ 

$$
f(g(w))

$$

$$
f(g)=g^2\\
g(w)=y-\^{y} \equiv y-wx+b\\ \\
$$

First, let’s differentiate each function

for $f$, we know that :

$$
\dfrac{\delta f}{\delta g} = 2g
$$



And for g :

$$
\delta g = \delta g(w)\delta b\\
\equiv \dfrac{\delta g}{\delta w} = \delta g(w)
$$

$$
\delta g = \delta (y - (wx+b))\delta w \\
\equiv \delta g = \delta (y -wx -b) \delta w \\
\equiv \delta g = -x\delta w \\
\equiv \dfrac{\delta g}{\delta w} = -x
$$

We have a composite function, we have to apply the chain rule (to differentiate the outer and inner functions) to find the derivative of $f$ with respect of $w$

$$
\delta f(g(w))= \delta f(g) \delta g(w) \delta w \\
\equiv \dfrac {\delta f(g(w))}{\delta w} = \delta f(g) \delta g(w) \\
\equiv \dfrac {\delta f(g(w))}{\delta w} = 2(y - wx+b)*(-x) \\
\equiv \dfrac {\delta f(g(w))}{\delta w} = -2x(y - wx+b) \\
$$


When you have two or more derivates of a function, they are called a gradient (hence Gradient Descent)

Our gradient for the Sum Squared Error is : 

$$
\dfrac {\delta f(g(b))}{\delta b} = -2(y - wx+b) \\
\dfrac {\delta f(g(w))}{\delta w} = -2x(y - wx+b) \\
$$


But our subject, being smart, doest things slightly differently : 

[Different Loss Functions](https://medium.com/@mlblogging.k/14-loss-functions-you-can-use-for-regression-b24db8dff987)

It uses a Mean Squared Error function with 2 smart elements : 

$$
J(w,b)=\dfrac{1}{2M}\displaystyle\sum_{i=1}^n((wx+b)-y)^2
$$


Instead of doing $|y -\^y |$ it does $|\^y-y |$ → That way when derived, the value is positive.

$$
\delta g = \delta ((wx+b) - y )\delta b \\
\equiv \delta g = 1\delta b \\
\equiv \dfrac{\delta g}{\delta b} = 1
$$

$$
\delta g = \delta ((wx+b) - y )\delta w \\
\equiv \delta g = x\delta w \\
\equiv \dfrac{\delta g}{\delta w} = x
$$



Then we multiply by $\dfrac{1}{2M}$ to cancel out the 2 in factor in $\delta f(g)$

so that 

$$
\dfrac{\delta J(w,b)}{\delta w}=\dfrac{1}{2M}\displaystyle\sum_{i=1}^n2(wx+b- y)\\
\equiv \dfrac{1}{M}\displaystyle\sum_{i=1}^n(wx+b - y)
$$

$$
\dfrac{\delta J(w,b)}{\delta b}=\dfrac{1}{2M}\displaystyle\sum_{i=1}^n2(wx+b- y)x\\
\equiv \dfrac{1}{M}\displaystyle\sum_{i=1}^n(wx+b - y)x
$$

That way we have easy functions to compute and now you know the origin function of this gradient !

[https://datascience.stackexchange.com/questions/10188/why-do-cost-functions-use-the-square-error](https://datascience.stackexchange.com/questions/10188/why-do-cost-functions-use-the-square-error)

[Statquest - Gradient descent](https://www.youtube.com/watch?v=sDv4f4s2SB8)
